# Chương 11: Bài tập 

## Mục tiêu thực hành

Bài thực hành này dựa trên **Chapter 11 - Deep learning for text**.

Sau bài thực hành, sinh viên cần làm được:

1. Chuẩn hóa văn bản, tokenization và lập vocabulary.
2. Dùng `TextVectorization` cho `int`, `multi_hot`, `count`, `tf_idf`.
3. Huấn luyện baseline **bag-of-bigrams** cho phân loại cảm xúc.
4. Tạo sequence input, embedding, padding và masking.
5. Tự cài đặt self-attention đơn giản bằng NumPy.
6. Hiểu positional embedding và Transformer encoder tối giản.
7. Tạo dữ liệu **sequence-to-sequence** bằng teacher forcing.
8. Tạo **causal mask** cho decoder.

## 0. Chuẩn bị môi trường

In [1]:
# Ẩn bớt log TensorFlow để notebook gọn hơn khi chạy.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Các thư viện chuẩn dùng cho regex, xử lý chuỗi và chuẩn hóa Unicode.
import re
import string
import unicodedata

# Các thư viện tính toán, xử lý bảng và vẽ biểu đồ.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# TensorFlow/Keras dùng để thực hành các layer xử lý văn bản trong Chapter 11.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization

# Cố định seed để kết quả huấn luyện demo ổn định hơn giữa các lần chạy.
np.random.seed(42)
tf.random.set_seed(42)

# Ưu tiên font hỗ trợ tiếng Việt cho các biểu đồ Matplotlib.
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Arial", "Tahoma", "Segoe UI", "DejaVu Sans"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

print("TensorFlow:", tf.__version__)
print("Matplotlib font:", plt.rcParams["font.family"])

I0000 00:00:1782301932.166573  101386 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782301933.104281  101386 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
Matplotlib font: ['Arial']


## 1. Dataset toy cho phân loại cảm xúc

In [2]:
# Dataset toy gồm 20 câu review phim.
# 10 câu đầu là positive, 10 câu sau là negative.
texts = np.array([
    "phim này rất hay",
    "diễn viên xuất sắc",
    "kịch bản cảm động",
    "tôi thích bộ phim này",
    "âm nhạc tuyệt vời",
    "phim vui và cuốn hút",
    "hình ảnh đẹp",
    "kết thúc làm tôi hài lòng",
    "một tác phẩm đáng xem",
    "nội dung thông minh",
    "phim này rất dở",
    "diễn viên tệ",
    "kịch bản nhàm chán",
    "tôi không thích bộ phim này",
    "âm nhạc khó chịu",
    "phim dài và buồn ngủ",
    "hình ảnh xấu",
    "kết thúc làm tôi thất vọng",
    "một tác phẩm không đáng xem",
    "nội dung rời rạc",
], dtype=object)

# Nhãn 1 là positive, nhãn 0 là negative.
labels = np.array([1]*10 + [0]*10)

# Tạo DataFrame để kiểm tra dữ liệu trực quan.
df = pd.DataFrame({
    "text": texts,
    "label": labels,
    "sentiment": np.where(labels == 1, "positive", "negative"),
})
df

,text,label,sentiment
0,phim này rất hay,1,positive
1,diễn viên xuất sắc,1,positive
2,kịch bản cảm động,1,positive
3,tôi thích bộ phim này,1,positive
4,âm nhạc tuyệt vời,1,positive
5,phim vui và cuốn hút,1,positive
6,hình ảnh đẹp,1,positive
7,kết thúc làm tôi hài lòng,1,positive
8,một tác phẩm đáng xem,1,positive
9,nội dung thông minh,1,positive


## 2. Bài 1 - Text standardization và tokenization

Hoàn thành hai hàm:

- `remove_accents(text)`: bỏ dấu Unicode, ví dụ `México -> Mexico`.
- `standardize_text(text)`: chuyển chữ thường, bỏ dấu câu, bỏ khoảng trắng thừa.

Sau đó dùng `tokenize_words(text)` để tách token theo khoảng trắng.

In [3]:
def remove_accents(text):
    """Bỏ dấu Unicode khỏi chuỗi văn bản.

    Tham số:
        text (str): Chuỗi đầu vào có thể chứa ký tự có dấu.

    Trả về:
        str: Chuỗi đã bỏ dấu. Sinh viên cần hoàn thành phần xử lý.
    """
    # TODO: dùng unicodedata.normalize("NFD", text)
    # Gợi ý: bỏ các ký tự có category == "Mn".
    normalized = unicodedata.normalize("NFD", text)

    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")


def standardize_text(text, strip_accents=True):
    """Chuẩn hóa văn bản trước khi tokenization.

    Tham số:
        text (str): Chuỗi văn bản đầu vào.
        strip_accents (bool): Nếu True thì bỏ dấu Unicode trước khi bỏ dấu câu.

    Trả về:
        str: Chuỗi đã chuẩn hóa. Sinh viên cần hoàn thành các bước bên dưới.
    """
    # TODO:
    # 1. Chuyển text sang chữ thường bằng lower().
    # 2. Nếu strip_accents=True thì gọi remove_accents(text).
    # 3. Bỏ punctuation bằng regex.
    # 4. Thay nhiều khoảng trắng thành 1 khoảng trắng và strip.
    text = text.lower()

    if strip_accents:
        text = remove_accents(text)

    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_words(text):
    """Tách văn bản thành danh sách word token.

    Tham số:
        text (str): Chuỗi văn bản đầu vào.

    Trả về:
        list[str]: Danh sách token sau khi chuẩn hóa nhẹ và tách theo khoảng trắng.
    """
    # Hàm này dùng standardize_text rồi split theo khoảng trắng.
    return standardize_text(text, strip_accents=False).split()


# Chạy thử trên vài câu để kiểm tra hàm chuẩn hóa và tokenization.
examples = [
    "Sunset came. I was staring at the México sky. Isn't nature splendid??",
    "Phim này RẤT hay!!!",
]

for s in examples:
    print("Gốc       :", s)
    print("Chuẩn hóa :", standardize_text(s))
    print("Tokens    :", tokenize_words(s))
    print()

Gốc       : Sunset came. I was staring at the México sky. Isn't nature splendid??
Chuẩn hóa : sunset came i was staring at the mexico sky isnt nature splendid
Tokens    : ['sunset', 'came', 'i', 'was', 'staring', 'at', 'the', 'méxico', 'sky', 'isnt', 'nature', 'splendid']

Gốc       : Phim này RẤT hay!!!
Chuẩn hóa : phim nay rat hay
Tokens    : ['phim', 'này', 'rất', 'hay']



## 3. Bài 2 - Vocabulary indexing

Xây vocabulary từ `texts` với quy ước:

- `""` có index `0`, dùng cho padding.
- `"[UNK]"` có index `1`, dùng cho token ngoài vocabulary.

Sau đó viết:

- `encode_text(text, vocab)`: text -> list chỉ số.
- `decode_ids(ids, inverse_vocab)`: list chỉ số -> text gần đúng.

In [5]:
# Vocabulary có hai token đặc biệt:
# "" dùng cho padding, "[UNK]" dùng cho token lạ.
vocab = {"": 0, "[UNK]": 1}

# TODO: duyệt qua texts, token hóa từng câu, thêm token mới vào vocab.
for text in examples:
    for token in tokenize_words(text):
        if token not in vocab:
            vocab[token] = len(vocab)

# Tạo bảng tra ngược index -> token để phục vụ bước decode.
inverse_vocab = {idx: token for token, idx in vocab.items()}


def encode_text(text, vocab):
    """Mã hóa một câu thành danh sách token indices.

    Tham số:
        text (str): Câu đầu vào cần mã hóa.
        vocab (dict[str, int]): Từ điển ánh xạ token sang index.

    Trả về:
        list[int]: Danh sách index; token lạ cần ánh xạ về vocab["[UNK]"].
    """
    # TODO: trả về list index, token lạ dùng vocab["[UNK]"].
    return [vocab.get(tok, vocab["[UNK]"]) for tok in tokenize_words(text)]


def decode_ids(ids, inverse_vocab):
    """Giải mã danh sách index thành chuỗi token gần đúng.

    Tham số:
        ids (Iterable[int]): Danh sách token indices.
        inverse_vocab (dict[int, str]): Từ điển ánh xạ index về token.

    Trả về:
        str: Chuỗi token sau khi bỏ padding 0.
    """
    # TODO: bỏ padding 0 và ghép token bằng khoảng trắng.
    tokens = [inverse_vocab[idx] for idx in ids if idx != 0]
    return " ".join(tokens)


# Câu có token "mới" nhiều khả năng chưa có trong vocabulary.
sample = "tôi rất thích phim mới"
ids = encode_text(sample, vocab)
print("Kích thước vocab:", len(vocab))
print("Sample:", sample)
print("Encoded:", ids)
print("Decoded:", decode_ids(ids, inverse_vocab))

Kích thước vocab: 18
Sample: tôi rất thích phim mới
Encoded: [1, 16, 1, 14, 1]
Decoded: [UNK] rất [UNK] phim [UNK]


## 4. Bài 3 - TextVectorization output modes

Dùng `TextVectorization` để so sánh 4 kiểu vector hóa:

1. `output_mode="int"`
2. `output_mode="multi_hot"`
3. `output_mode="count"`
4. `output_mode="tf_idf"` với `ngrams=2`

In ra vocabulary và vector của câu mẫu.

In [6]:
# Câu mẫu dùng để so sánh nhiều output_mode của TextVectorization.
sample_batch = tf.constant(["tôi rất thích phim này"])

# TODO: tạo 4 TextVectorization layer theo yêu cầu trong đề bài.
vectorizers = {
    "int": TextVectorization(output_mode="int"),
    "multi_hot": TextVectorization(output_mode="multi_hot"),
    "count": TextVectorization(output_mode="count"),
    "tf_idf bigram": TextVectorization(output_mode="tf_idf", ngrams=2),
}

for name, layer in vectorizers.items():
    # TODO: gọi layer.adapt(texts) để layer học vocabulary/thống kê từ dữ liệu.
    layer.adapt(texts)
    # TODO: vector hóa sample_batch bằng layer(sample_batch).
    vectorized_sample = layer(sample_batch)
    vocab = layer.get_vocabulary()
    
    print("\n==", name, "==")
    print("Vocabulary:", vocab)
    print("Vector:",  vectorized_sample)


== int ==
Vocabulary: ['', '[UNK]', np.str_('phim'), np.str_('tôi'), np.str_('này'), np.str_('ảnh'), np.str_('đáng'), np.str_('âm'), np.str_('xem'), np.str_('và'), np.str_('viên'), np.str_('tác'), np.str_('thúc'), np.str_('thích'), np.str_('rất'), np.str_('phẩm'), np.str_('nội'), np.str_('nhạc'), np.str_('một'), np.str_('làm'), np.str_('kịch'), np.str_('kết'), np.str_('không'), np.str_('hình'), np.str_('dung'), np.str_('diễn'), np.str_('bộ'), np.str_('bản'), np.str_('động'), np.str_('đẹp'), np.str_('xấu'), np.str_('xuất'), np.str_('vời'), np.str_('vọng'), np.str_('vui'), np.str_('tệ'), np.str_('tuyệt'), np.str_('thất'), np.str_('thông'), np.str_('sắc'), np.str_('rời'), np.str_('rạc'), np.str_('nhàm'), np.str_('ngủ'), np.str_('minh'), np.str_('lòng'), np.str_('khó'), np.str_('hút'), np.str_('hài'), np.str_('hay'), np.str_('dở'), np.str_('dài'), np.str_('cảm'), np.str_('cuốn'), np.str_('chịu'), np.str_('chán'), np.str_('buồn')]
Vector: tf.Tensor([[ 3 14 13  2  4]], shape=(1, 5), dtype=i

E0000 00:00:1782305304.415371  101386 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


## 5. Bài 4 - Bag-of-bigrams classifier

Huấn luyện mô hình phân loại cảm xúc đơn giản:

```text
raw string -> TextVectorization(ngrams=2, multi_hot) -> Dense -> Dropout -> Dense(sigmoid)
```

Yêu cầu:

- Dùng `TextVectorization(max_tokens=100, ngrams=2, output_mode="multi_hot")`.
- Train khoảng 60-100 epochs vì dataset rất nhỏ.
- In accuracy cuối và dự đoán 4 câu mới.

In [7]:
# Vectorizer cho baseline bag-of-bigrams.
bow_vectorizer = TextVectorization(
    # TODO: max_tokens=100, ngrams=2, output_mode="multi_hot"
    max_tokens=100,
    ngrams=2,
    output_mode="multi_hot",
)

# TODO: adapt vectorizer trên texts.
bow_vectorizer.adapt(texts)

# Input là raw string để mô hình có thể nhận trực tiếp câu review.
inputs = keras.Input(shape=(), dtype=tf.string)

# TODO: điền pipeline model:
# 1. x = bow_vectorizer(inputs)
# 2. Dense hidden layer
# 3. Dropout nếu muốn
# 4. Dense sigmoid output
x = bow_vectorizer(inputs)
x = layers.Dense(8, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
bow_model = keras.Model(inputs, outputs)

# Binary classification nên dùng binary_crossentropy và sigmoid.
bow_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# TODO: fit model trên texts và labels.
history = bow_model.fit(texts, labels, epochs=80, verbose=0)

print("Train accuracy cuối:", round(history.history["accuracy"][-1], 3))

# Các câu mới để kiểm tra mô hình sau khi train.
new_reviews = np.array([
    "phim rất đáng xem",
    "tôi không thích diễn viên",
    "kịch bản thông minh và cảm động",
    "phim buồn ngủ và rời rạc",
], dtype=object)

# TODO: predict và tạo DataFrame kết quả.
probs = bow_model.predict(new_reviews, verbose=0).reshape(-1)

pd.DataFrame({
    "review": new_reviews,
    "P(positive)": np.round(probs, 3),
})

Train accuracy cuối: 0.95


,review,P(positive)
0,phim rất đáng xem,0.471
1,tôi không thích diễn viên,0.280
2,kịch bản thông minh và cảm động,0.515
3,phim buồn ngủ và rời rạc,0.484


## 6. Bài 5 - Sequence input, Embedding, Padding, Masking

Tạo `TextVectorization(output_mode="int", output_sequence_length=8)`, sau đó:

- Vector hóa 3 câu đầu tiên.
- Tạo `Embedding(input_dim=50, output_dim=6, mask_zero=True)`.
- In shape của integer sequence, embedding output và mask.

In [8]:
# Vectorizer dạng int giữ thứ tự token để dùng cho sequence model.
seq_vectorizer = TextVectorization(
    # TODO: max_tokens=50, output_mode="int", output_sequence_length=8
    max_tokens=50,
    output_mode="int",
    output_sequence_length=8,
)

# TODO: adapt seq_vectorizer trên texts.
seq_vectorizer.adapt(texts)
encoded_batch = seq_vectorizer(tf.constant(texts[:3]))

# Embedding biến token index thành vector dense.
embedding = layers.Embedding(
    # TODO: input_dim=50, output_dim=6, mask_zero=True
    input_dim=50, 
    output_dim=6, 
    mask_zero=True
)

# TODO: gọi embedding(encoded_batch) và compute_mask(encoded_batch).
embedded_batch = embedding(encoded_batch)
mask = embedding.compute_mask(encoded_batch)

print("Encoded batch:")
print(encoded_batch)
print("Embedding output shape:", "TODO")
print("Mask:")
print(mask)

Encoded batch:
tf.Tensor(
[[ 2  4 14 49  0  0  0  0]
 [25 10 31 39  0  0  0  0]
 [20 27  1 28  0  0  0  0]], shape=(3, 8), dtype=int64)
Embedding output shape: TODO
Mask:
tf.Tensor(
[[ True  True  True  True False False False False]
 [ True  True  True  True False False False False]
 [ True  True  True  True False False False False]], shape=(3, 8), dtype=bool)


## 7. Bài 6 - Self-attention bằng NumPy

Cài đặt self-attention đơn giản cho một sequence vector:

1. Tính scores: `scores = X @ X.T`.
2. Scale bằng `sqrt(d)`.
3. Softmax theo từng dòng.
4. Tính output: `weights @ X`.

Sau đó quan sát attention weight của token `station`.

In [11]:
def row_softmax(x):
    """Tính softmax theo từng dòng của ma trận điểm số.

    Tham số:
        x (np.ndarray): Ma trận scores shape `(T, T)`.

    Trả về:
        np.ndarray: Ma trận xác suất cùng shape, mỗi dòng có tổng bằng 1.
    """
    # TODO: softmax theo từng dòng.
    x = x - np.max(x)
    exp = np.exp(x)
    return exp / np.sum(exp)


def self_attention_numpy(X):
    """Tính self-attention đơn giản bằng NumPy.

    Tham số:
        X (np.ndarray): Ma trận token vectors shape `(sequence_length, dim)`.

    Trả về:
        tuple[np.ndarray, np.ndarray]: output vectors và attention weights.
    """
    # TODO:
    d = X.shape[-1]
    scores = X @ X.T / np.sqrt(d)
    weights = row_softmax(scores)
    output = weights @ X
    return output, weights


# Sequence toy để quan sát attention của token "station".
sentence = ["the", "train", "left", "the", "station", "on", "time"]
X = np.array([
    [0.1, 0.0, 0.0],
    [0.9, 0.8, 0.1],
    [0.2, 0.3, 0.8],
    [0.1, 0.0, 0.0],
    [0.8, 0.9, 0.1],
    [0.1, 0.2, 0.2],
    [0.2, 0.1, 0.7],
], dtype="float32")

output, weights = self_attention_numpy(X)

pivot_index = sentence.index("station")

# TODO: tạo DataFrame hiển thị attention weights của token station.
pivot = X[pivot_index]

scores = X @ pivot

weights = row_softmax(scores)

context_aware_vector = weights @ X

attention_table = pd.DataFrame({
    "token": sentence,
    "dot_score_with_station": np.round(scores, 3),
    "attention_weight": np.round(weights, 3),
})

print("Context-aware vector cho token 'station':", np.round(context_aware_vector, 3))
attention_table

Context-aware vector cho token 'station': [0.546 0.542 0.226]


,token,dot_score_with_station,attention_weight
0,the,0.08,0.072
1,train,1.45,0.282
2,left,0.51,0.110
3,the,0.08,0.072
4,station,1.46,0.285
5,on,0.28,0.088
6,time,0.32,0.091


## 8. Bài 7 - PositionalEmbedding và TransformerEncoder tối giản

Hoàn thành class `PositionalEmbedding`:

```text
output = token_embedding(inputs) + position_embedding(positions)
```

Sau đó kiểm tra output shape.

In [13]:
class PositionalEmbedding(layers.Layer):
    """Layer cộng token embedding với positional embedding.

    Sinh viên cần hoàn thành layer này để Transformer biết thông tin vị trí token.
    """

    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        """Khởi tạo token embedding và position embedding.

        Tham số:
            sequence_length (int): Độ dài sequence tối đa.
            input_dim (int): Kích thước vocabulary.
            output_dim (int): Số chiều embedding.
            **kwargs: Tham số bổ sung cho lớp `layers.Layer`.
        """
        super().__init__(**kwargs)
        # TODO: tạo self.token_embeddings và self.position_embeddings.
        self.token_embeddings = layers.Embedding(input_dim=input_dim, output_dim=output_dim, mask_zero=True)

        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        """Tạo embedding có thông tin vị trí.

        Tham số:
            inputs (tf.Tensor): Tensor token index shape `(batch, sequence_length)`.

        Trả về:
            tf.Tensor: Tensor embedding sau khi cộng token embedding và position embedding.
        """
        # TODO: tạo positions bằng tf.range.
        length = tf.shape(inputs)[-1]
        # TODO: trả về token embedding + position embedding.
        positions = tf.range(start=0, limit=length, delta=1)

        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)

        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        """Tạo mask cho token padding.

        Tham số:
            inputs (tf.Tensor): Tensor token index, trong đó 0 là padding.
            mask (tf.Tensor | None): Mask đầu vào nếu có.

        Trả về:
            tf.Tensor: Boolean mask, True cho token thật và False cho padding.
        """
        # TODO: trả về True cho token khác 0.
        return tf.math.not_equal(inputs, 0)


pos_layer = PositionalEmbedding(sequence_length=8, input_dim=50, output_dim=6)
# TODO: gọi pos_layer(encoded_batch).
pos_output = pos_layer(encoded_batch)
print("Positional embedding output shape:", pos_output.shape)

Positional embedding output shape: (3, 8, 6)


## 9. Bài 8 - Sequence-to-sequence teacher forcing và causal mask

Cho các cặp câu English-Spanish toy. Hãy:

1. Thêm `[start]` và `[end]` vào target Spanish.
2. Vector hóa source và target.
3. Tạo `decoder_inputs = target[:, :-1]`.
4. Tạo `target_outputs = target[:, 1:]`.
5. Viết hàm tạo causal mask kích thước `T x T`.

In [16]:
# Các cặp câu English-Spanish toy cho bài sequence-to-sequence.
pairs = [
    ("how is the weather today", "qué tiempo hace hoy"),
    ("i like this movie", "me gusta esta película"),
    ("she can play piano", "ella puede tocar piano"),
    ("this book is good", "este libro es bueno"),
]

source_texts = [src for src, tgt in pairs]
target_texts = [
    # TODO: thêm [start] và [end] vào từng câu target Spanish.
    "[start] " + tgt + " [end]" for src, tgt in pairs
]

# Giữ lại dấu [ ] trong [start]/[end], nhưng bỏ punctuation khác.
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "").replace("]", "")


def custom_standardization(input_string):
    """Chuẩn hóa target Spanish cho TextVectorization.

    Tham số:
        input_string (tf.Tensor): Tensor chuỗi đầu vào.

    Trả về:
        tf.Tensor: Chuỗi đã lower-case và bỏ punctuation không cần thiết.
    """
    # Dùng tf.strings để hàm tương thích với TextVectorization.
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, f"[{re.escape(strip_chars)}]", "")


# Vectorizer riêng cho source và target vì hai ngôn ngữ có vocabulary khác nhau.
source_vectorization = TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=8,
)
target_vectorization = TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=9,
    standardize=custom_standardization,
)

# TODO: adapt 2 vectorizer.
source_vectorization = TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=8,
)
target_vectorization = TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=9,
    standardize=custom_standardization,
)

source_vectorization.adapt(source_texts)
target_vectorization.adapt(target_texts)

# TODO: tạo source_encoded, target_encoded.
source_encoded = source_vectorization(tf.constant(source_texts))
target_encoded = target_vectorization(tf.constant(target_texts))

# TODO: tạo decoder_inputs = target_encoded[:, :-1].
decoder_inputs = target_encoded[:, :-1]

# TODO: tạo target_outputs = target_encoded[:, 1:].
target_outputs = target_encoded[:, 1:]

def causal_attention_mask(size):
    """Tạo causal mask cho decoder.

    Tham số:
        size (int): Độ dài target sequence.

    Trả về:
        tf.Tensor: Ma trận mask shape `(size, size)`, 1 cho vị trí được nhìn
        và 0 cho vị trí tương lai bị che.
    """
    # TODO: return ma trận 1 nếu j <= i, 0 nếu j > i.
    i = tf.range(size)[:, tf.newaxis]
    j = tf.range(size)[tf.newaxis, :]

    return tf.cast(i >= j, dtype="int32")


print("source_encoded shape :", source_encoded.shape)
print("decoder_inputs shape :", decoder_inputs.shape)
print("target_outputs shape :", target_outputs.shape)
print(causal_attention_mask(8))

source_encoded shape : (4, 8)
decoder_inputs shape : (4, 8)
target_outputs shape : (4, 8)
tf.Tensor(
[[1 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0]
 [1 1 1 1 0 0 0 0]
 [1 1 1 1 1 0 0 0]
 [1 1 1 1 1 1 0 0]
 [1 1 1 1 1 1 1 0]
 [1 1 1 1 1 1 1 1]], shape=(8, 8), dtype=int32)


## 10. Câu hỏi thảo luận cuối buổi

Trả lời ngắn gọn:

1. Khi nào nên thử **bag-of-bigrams** trước sequence model?

    Khi dữ liệu ít, cần mô hình đơn giản làm baseline nhanh hoặc tài nguyên tính toán hạn chế.

2. Vì sao one-hot sequence thường tốn kém hơn embedding?

    One-hot tạo ma trận thưa khổng lồ, còn embedding nén thành các vector liên tục, dày đặc.

3. Self-attention khác RNN ở điểm nào khi xử lý quan hệ xa trong câu?

    Self-attention kết nối trực tiếp mọi từ, còn RNN phải truyền tuần tự qua từng bước bước.

4. Vì sao Transformer cần positional embedding?

    Transformer xử lý từ song song nên cần positional embedding để nhận biết thứ tự trước sau.

5. Vì sao decoder cần causal mask?

    Causal mask ngăn decoder nhìn trước các từ tương lai, đảm bảo tính tự hồi quy.